# Chapter 10 · Gradient descent by hand
**Deck 2 · Live demo.** The machine starts at w = 0, b = 0 and finds the rule on its own, one step at a time.

**You will:** compute one step by hand · loop it in NumPy · watch w overshoot and come back · prove it ends at w = 5, b = 40.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(precision=3, suppress=True)
plt.rcParams.update({"figure.figsize": (8, 4.5), "axes.grid": True, "font.size": 12})

## 1 · Your prediction
How many steps will gradient descent need (with step size 0.01) until Asha’s prediction is within half a mark of 80?

In [ ]:
my_prediction_steps = 1600   # ← your guess

In [ ]:
hours = np.array([2., 4., 6.])
marks = np.array([50., 60., 70.])

def loss(w, b):
    return np.mean((w * hours + b - marks) ** 2)

## 2 · The slopes
One slope per knob (Chapter 9):

$$\frac{\partial L}{\partial w} = \frac{2}{n}\sum(\hat{y}-y)\,x \qquad \frac{\partial L}{\partial b} = \frac{2}{n}\sum(\hat{y}-y)$$

In [ ]:
def slopes(w, b):
    errors = w * hours + b - marks
    return 2 * np.mean(errors * hours), 2 * np.mean(errors)

print('slopes at w=0, b=0:', slopes(0, 0))

## 3 · One step by hand
$$w \leftarrow w - \eta \cdot \frac{\partial L}{\partial w}$$ with step size η = 0.01.

In [ ]:
eta = 0.01
gw, gb = slopes(0, 0)
w1, b1 = 0 - eta * gw, 0 - eta * gb
print(f'step 1: w = {w1:.2f}, b = {b1:.2f}, loss {loss(0, 0):,.1f} → {loss(w1, b1):,.1f}')

## 4 · The loop

In [ ]:
def descend(eta, steps, w=0.0, b=0.0):
    history = [(w, b, loss(w, b))]
    for _ in range(steps):
        gw, gb = slopes(w, b)
        w, b = w - eta * gw, b - eta * gb
        history.append((w, b, loss(w, b)))
    return np.array(history)

h = descend(0.01, 3000)
w_end, b_end = h[-1, 0], h[-1, 1]
print(f'after 3,000 steps: w = {w_end:.3f}, b = {b_end:.3f}, Asha → {8 * w_end + b_end:.2f}')

In [ ]:
def steps_to_asha(eta, limit=100_000):
    w = b = 0.0
    for i in range(1, limit):
        gw, gb = slopes(w, b)
        w, b = w - eta * gw, b - eta * gb
        if not np.isfinite(w) or abs(w) > 1e9:
            return None
        if abs(8 * w + b - 80) < 0.5 and abs(w - 5) < 0.1 and abs(b - 40) < 0.5:
            return i
steps = steps_to_asha(0.01)
print('steps to reach Asha ≈ 80:', steps, '· your guess:', my_prediction_steps)

## 5 · PROOF

In [ ]:
assert np.allclose(slopes(0, 0), (-506.6667, -120.0), atol=1e-3), 'slopes −506.7 and −120'
assert round(w1, 2) == 5.07 and round(b1, 2) == 1.20, 'step 1: w = 5.07, b = 1.20'
assert round(loss(0, 0), 1) == 3666.7 and round(loss(w1, b1), 1) == 1484.8, 'loss 3,666.7 → 1,484.8'
assert h[:20, 0].max() > 12, 'w overshoots past 12'
assert abs(w_end - 5) < 0.01 and abs(b_end - 40) < 0.05, 'ends at w = 5, b = 40'
assert steps == 1604, 'the slide: about 1,604 steps'
print('✓ proof: every slide number reproduced')

## 6 · Picture: fast at first, then slow; w overshoots

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
ax[0].semilogy(h[:, 2]); ax[0].set_title('loss (log scale)'); ax[0].set_xlabel('step')
ax[1].plot(h[:, 0], label='w'); ax[1].plot(h[:, 1], label='b'); ax[1].axhline(5, ls=':'); ax[1].axhline(40, ls=':')
ax[1].set_title('w races ahead and overshoots; b crawls up to 40'); ax[1].set_xlabel('step'); ax[1].legend()
plt.show()

## 7 · Break it
Flip the minus sign to a plus: step *up* the slope. What happens?

In [ ]:
w = b = 0.0
for i in range(5):
    gw, gb = slopes(w, b)
    w, b = w + 0.01 * gw, b + 0.01 * gb   # wrong way!
    print(f'step {i + 1}: loss {loss(w, b):,.0f}')

## 8 · Your turn
Start from **w = 10, b = 100**. Does it still end at 5 and 40?

In [ ]:
h2 = descend(0.01, 3000, w=10, b=100)
print(h2[-1])

## Recap
- Slopes: 2/n · Σ error × hours (w) and 2/n · Σ error (b)
- Update: w ← w − η × slope, repeated
- Step 1: 5.07, 1.20 · after ~1,600 steps: Asha ≈ 80 · ends at w = 5, b = 40